# SSU Data Dash: quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/erwinzhang7/SSU-Data-Dash/blob/main/notebooks/quickstart.ipynb)

The same steps as `python run.py validate / report / predict / check`, one cell at a time.
Run the cells from top to bottom. Colab already has every package this needs.

In [3]:
# Setup. In Colab this downloads the starter code; on your own computer it just
# moves to the repo folder.
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SSU-Data-Dash"):
        !git clone -q https://github.com/erwinzhang7/SSU-Data-Dash.git /content/SSU-Data-Dash
    os.chdir("/content/SSU-Data-Dash")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

# Reload features.py and models.py automatically after you edit them.
#%load_ext autoreload
#%autoreload 2

In [6]:

import os

folder = "/content/SSU-Data-Dash"

for name in ["features.py", "models.py"]:
    path = os.path.join(folder, name)
    print(name, ":", os.path.getsize(path), "bytes")

with open("/content/SSU-Data-Dash/models.py", "r") as f:
    print(f.read()[:1500])


features.py : 2616 bytes
models.py : 2073 bytes
"""Improved, leakage-aware models for the two SSU Data Dash tracks.
Drop-in replacement for official models.py; compatible with run.py.
"""
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
import features


def make_model(track):
    if track == 'off_market':
        return PriceModel(features.off_market_features, list_aware=False)
    if track == 'list_aware':
        return PriceModel(features.list_aware_features, list_aware=True)
    raise ValueError(f'Unknown track: {track}')


class PriceModel:
    def __init__(self, feature_function, list_aware=False):
        self.feature_function = feature_function
        self.list_aware = list_aware

    def fit(self, train):
        X = self.feature_function(train)
        categorical = [c for c

In [7]:

import os
import sys

os.chdir("/content/SSU-Data-Dash")

# 清除之前加载的旧版模块
for module in ["features", "models", "run"]:
    sys.modules.pop(module, None)

# 运行比赛验证
import run
run.validate()


listings.parquet: reconstructing file:   0%|          |  0.00B / 32.9MB            

listings.parquet: downloading bytes:           |  0.00B            

2025-12: trained on 139,931 earlier sales, predicted 1,238 homes
2026-01: trained on 141,169 earlier sales, predicted 1,058 homes
2026-02: trained on 142,227 earlier sales, predicted 1,442 homes
2026-03: trained on 143,669 earlier sales, predicted 1,816 homes
2026-04: trained on 145,485 earlier sales, predicted 2,224 homes
2026-05: trained on 147,709 earlier sales, predicted 2,291 homes
Wrote outputs/validation_predictions.csv


In [8]:

import run
run.report()


# Validation report

Walk-forward validation, 2025-12 to 2026-05: each month was predicted by models trained only on the sales before it.
Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.

| Track | Homes | MAPE | MdAPE | Median signed error |
|---|---|---|---|---|
| off_market | 10,069 | 14.19% (13.95 to 14.44) | 11.07% (10.85 to 11.36) | +0.81% |
| list_aware | 10,069 | 5.55% (5.46 to 5.64) | 4.45% (4.36 to 4.56) | +0.85% |

MAPE: mean absolute percentage error. MdAPE: median absolute percentage error.
A median signed error above 0 means the model tends to predict too high.
The intervals treat homes as independent. Homes sold in the same month share the same
market swings, so the real uncertainty is somewhat wider.

![Error by price range](../figures/error_by_price_range.png)

Wrote reports/validation_report.md, reports/validation_report.json, reports/results_table.tex and figures/error_by_price_range.png


## 1. Get the data

The files download from Hugging Face the first time and are cached for later runs. If you downloaded them by hand, set `DATA_DIR` in the next cell instead.

In [ ]:
# os.environ["DATA_DIR"] = "/path/to/folder"   # only if you downloaded the files yourself

import data
listings = data.load("listings")
print(f"{len(listings):,} sales, {listings['date_sold'].min():%B %Y} to {listings['date_sold'].max():%B %Y}")
listings.head()

In [ ]:
off_market = data.load("off_market")
list_aware = data.load("list_aware")
print(f"{len(off_market):,} homes to value with no list price (off-market)")
print(f"{len(list_aware):,} homes to value that have a list price (list-aware)")
off_market.head()

In [ ]:
# A first look: the median sold price each month, by property type.
monthly = listings.groupby([listings["date_sold"].dt.to_period("M"), "house_type_name"])["sold_price"].median()
monthly.unstack().plot(figsize=(9, 4), ylabel="Median sold price ($)", xlabel="");

## 2. Validate

Each month from December 2025 to May 2026 is predicted by models trained only on the sales
before it, the same way the real test works. This takes about a minute.

In [ ]:
import run
run.validate()

In [ ]:
run.report()
from IPython.display import Image
Image("figures/error_by_price_range.png")

## 3. Improve the models

Edit **features.py** and **models.py**, then rerun the two cells above to see whether your
validation error goes down. In Colab, open them from the folder icon on the left
(`SSU-Data-Dash`), edit, and save with Ctrl+S (Cmd+S on a Mac).
Colab deletes your files when the session ends, so download anything you change.

## 4. Predict and check

This fits the models on all 150,000 sales and writes the two submission files.

In [ ]:
run.predict()
run.check()

In [ ]:
# In Colab: download the two submission files.
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("submissions/submission_off_market.csv")
    files.download("submissions/submission_list_aware.csv")